# AirSense AI: Urban Air Quality Intelligence Platform for Delhi
### End-to-End Machine Learning Pipeline
This notebook details the machine learning workflow for predicting 24/48/72-hour AQI in Delhi, explaining predictions using SHAP, and saving models for deployment. All training data represents real measurements from Delhi CPCB stations, Open-Meteo weather records, Sentinel-5P TROPOMI satellite data, and OpenStreetMap geospatial layers.

In [ ]:
# Import core libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import shap
import pickle
import json
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold

# Style settings
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12

print("All libraries imported successfully!")

## 1. Data Collection & Source Descriptions
Our data is compiled from five primary real-world sources:
1. **CPCB (Central Pollution Control Board) & OpenAQ**: Hourly ground station air quality measurements (PM2.5, PM10, NO2, SO2, CO, O3, AQI).
2. **Open-Meteo Archive API**: Hourly meteorological conditions for Delhi (temperature, relative humidity, wind speed, wind direction, precipitation).
3. **Sentinel-5P Satellite (TROPOMI)**: Daily tropospheric column densities of NO2, CO, and SO2, representing satellite-observed air pollution.
4. **OpenStreetMap (OSM)**: Spatial markers for schools, hospitals, primary roads, and industrial areas.

Let's execute the data preparation scripts (if not already run) and load the merged dataset.

In [ ]:
# Run the merge script if the dataset doesn't exist
merged_path = "data/processed/merged_delhi_aqi.csv"
if not os.path.exists(merged_path):
    print("Merged dataset not found. Running merge script...")
    from scripts.merge_datasets import merge_datasets
    df = merge_datasets()
else:
    df = pd.read_csv(merged_path)

df['timestamp'] = pd.to_datetime(df['timestamp'])
print(f"Merged Dataset loaded. Shape: {df.shape}")
print(f"Stations: {df['station_name'].unique()}")
df.head()

## 2. Data Cleaning & Preprocessing
We inspect data quality, check for duplicate rows, search for missing values, and handle temporal alignments.

In [ ]:
print("Checking for duplicate timestamps per station:")
duplicates = df.duplicated(subset=['station_name', 'timestamp']).sum()
print(f"Duplicate rows: {duplicates}")

print("\nMissing values count per column:")
print(df.isnull().sum())

# Summary statistics
df.describe().T

## 3. Exploratory Data Analysis (EDA)
Let's visualize the temporal patterns of AQI, pollutant relationships, and meteorological impacts.

In [ ]:
# 1. Overall AQI Distribution across stations
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='station_name', y='aqi', palette='viridis')
plt.xticks(rotation=45, ha='right')
plt.title('AQI Distribution by Station in Delhi')
plt.tight_layout()
plt.show()

# 2. Correlation Heatmap
plt.figure(figsize=(12, 8))
numerical_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numerical_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Matrix of AQI, Weather, and Satellite Columns')
plt.tight_layout()
plt.show()

# 3. Monthly AQI Trends (showing winters vs summers)
df['month'] = df['timestamp'].dt.month
plt.figure(figsize=(10, 5))
sns.lineplot(data=df, x='month', y='aqi', marker='o', color='crimson')
plt.title('Average AQI Trend by Month in Delhi')
plt.xticks(range(1, 13))
plt.show()

## 4. Feature Engineering
To model the time-series forecasting problem, we construct features containing:
- **Lag Features**: Past readings of AQI and key pollutants (t-1, t-2, t-24, t-48).
- **Rolling Averages**: 6-hour, 12-hour, and 24-hour moving averages and standard deviations to capture recent trend gradients.
- **Wind Vectors (Geospatial ML)**: Decomposing wind speed and circular direction into U and V components (`wind_u` and `wind_v`).
- **Date/Time Attributes**: Hour, month, day of week, and weekend indicators.
- **Spatial Features**: Binned distances to OSM elements.
- **Satellite Pollution Columns**: Daily Sentinel-5P columns of NO2, CO, and SO2.

We will also prepare the targets: **24-hour**, **48-hour**, and **72-hour ahead** AQI predictions.

In [ ]:
def engineer_features(df):
    df = df.copy()
    df = df.sort_values(by=['station_name', 'timestamp'])
    
    # 1. Date/Time Features
    df['hour'] = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    df['month'] = df['timestamp'].dt.month
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    
    # 2. Wind U and V Vectors
    rads = np.radians(df['wind_direction'])
    df['wind_u'] = df['wind_speed'] * np.cos(rads)
    df['wind_v'] = df['wind_speed'] * np.sin(rads)
    
    # 3. Lag Features (AQI and PM2.5)
    for col in ['aqi', 'pm25', 'no2']:
        for lag in [1, 2, 24, 48]:
            df[f'{col}_lag_{lag}'] = df.groupby('station_name')[col].shift(lag)
            
    # 4. Rolling Features
    for col in ['aqi', 'pm25']:
        for window in [6, 12, 24]:
            df[f'{col}_roll_mean_{window}'] = df.groupby('station_name')[col].transform(lambda x: x.shift(1).rolling(window).mean())
            df[f'{col}_roll_std_{window}'] = df.groupby('station_name')[col].transform(lambda x: x.shift(1).rolling(window).std())
            
    # 5. Future Targets (24h, 48h, 72h ahead predictions)
    df['target_24h'] = df.groupby('station_name')['aqi'].shift(-24)
    df['target_48h'] = df.groupby('station_name')['aqi'].shift(-48)
    df['target_72h'] = df.groupby('station_name')['aqi'].shift(-72)
    
    return df

df_feat = engineer_features(df)
# Drop rows with NaN targets/lags resulting from shifts
df_clean = df_feat.dropna().reset_index(drop=True)
print(f"Feature matrix compiled. Shape: {df_clean.shape}")
df_clean.head()

## 5. Model Training
We train three separate **LightGBM Regressor** models to predict 24h, 48h, and 72h AQI directly. This avoids compounding errors of recursive models. We split train/test chronologically to represent a true production inference scenario.

In [ ]:
# Define feature columns
feature_cols = [
    'latitude', 'longitude', 'pm25', 'pm10', 'no2', 'so2', 'co', 'o3',
    'temperature', 'humidity', 'wind_speed', 'wind_u', 'wind_v', 'precipitation',
    's5p_no2_column', 's5p_co_column', 's5p_so2_column',
    'dist_to_hospital_km', 'dist_to_school_km', 'dist_to_industrial_km', 'dist_to_road_km',
    'hour', 'day_of_week', 'month', 'is_weekend',
    'aqi_lag_1', 'aqi_lag_2', 'aqi_lag_24', 'aqi_lag_48',
    'pm25_lag_1', 'pm25_lag_2', 'pm25_lag_24', 'pm25_lag_48',
    'no2_lag_1', 'no2_lag_2', 'no2_lag_24', 'no2_lag_48',
    'aqi_roll_mean_6', 'aqi_roll_std_6', 'aqi_roll_mean_12', 'aqi_roll_std_12', 'aqi_roll_mean_24', 'aqi_roll_std_24',
    'pm25_roll_mean_6', 'pm25_roll_std_6', 'pm25_roll_mean_12', 'pm25_roll_std_12', 'pm25_roll_mean_24', 'pm25_roll_std_24'
]

# Time-based train-test split (Chronological split)
# We split the data: 85% Train, 15% Test
split_idx = int(len(df_clean) * 0.85)
train_df = df_clean.iloc[:split_idx]
test_df = df_clean.iloc[split_idx:]

X_train, y_train_24h, y_train_48h, y_train_72h = train_df[feature_cols], train_df['target_24h'], train_df['target_48h'], train_df['target_72h']
X_test, y_test_24h, y_test_48h, y_test_72h = test_df[feature_cols], test_df['target_24h'], test_df['target_48h'], test_df['target_72h']

print(f"Training instances: {len(X_train)}")
print(f"Testing instances: {len(X_test)}")

# Hyperparameters optimized for small tabular data
lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 6,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1
}

print("\nTraining 24h prediction model...")
model_24h = lgb.train(lgb_params, lgb.Dataset(X_train, label=y_train_24h), num_boost_round=150)

print("Training 48h prediction model...")
model_48h = lgb.train(lgb_params, lgb.Dataset(X_train, label=y_train_48h), num_boost_round=150)

print("Training 72h prediction model...")
model_72h = lgb.train(lgb_params, lgb.Dataset(X_train, label=y_train_72h), num_boost_round=150)

print("Model training completed.")

## 6. Model Evaluation
We calculate standard error metrics: Root Mean Squared Error (RMSE), Mean Absolute Error (MAE), and Coefficient of Determination ($R^2$), and graph predictions against target actual labels.

In [ ]:
def evaluate_predictions(y_true, y_pred, horizon):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"[{horizon} Horizon] RMSE: {rmse:.2f} | MAE: {mae:.2f} | R2 Score: {r2:.2f}")
    return rmse, mae, r2

pred_24h = model_24h.predict(X_test)
pred_48h = model_48h.predict(X_test)
pred_72h = model_72h.predict(X_test)

evaluate_predictions(y_test_24h, pred_24h, "24h")
evaluate_predictions(y_test_48h, pred_48h, "48h")
evaluate_predictions(y_test_72h, pred_72h, "72h")

# Visualizing predictions vs actual
plt.figure(figsize=(14, 5))
plt.plot(y_test_24h.values[:120], label="Actual 24h AQI", color='blue', alpha=0.8)
plt.plot(pred_24h[:120], label="Predicted 24h AQI", color='orange', linestyle='--', alpha=0.9)
plt.title("AQI 24-Hour Horizon predictions vs Actuals (First 5 Days of Test Set)")
plt.ylabel("AQI")
plt.legend()
plt.show()

## 7. SHAP Explainability
We initialize a SHAP `TreeExplainer` on the 24h model to construct global feature importances and showcase a local sample force explanation.

In [ ]:
# Initialize SHAP Explainer
explainer_24h = shap.TreeExplainer(model_24h)
shap_values = explainer_24h(X_test)

# Global SHAP summary plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
plt.title("Global Feature Importance (SHAP Mean Absolute Impact)")
plt.tight_layout()
plt.show()

# Detailed SHAP Beeswarm summary plot
plt.figure(figsize=(12, 8))
shap.plots.beeswarm(shap_values, max_display=15, show=False)
plt.title("SHAP Feature Impact Beeswarm Plot (24h model)")
plt.tight_layout()
plt.show()

## 8. Save Model
We export the models and feature definitions so they can be loaded by the FastAPI application backend.

In [ ]:
os.makedirs("backend/trained_models", exist_ok=True)

# Export models to standard LightGBM formats
model_24h.save_model("backend/trained_models/lgbm_aqi_24h.txt")
model_48h.save_model("backend/trained_models/lgbm_aqi_48h.txt")
model_72h.save_model("backend/trained_models/lgbm_aqi_72h.txt")

# Save features metadata
metadata = {
    "features": feature_cols,
    "training_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

with open("backend/trained_models/model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)
    
print("Trained models and feature metadata exported successfully!")

## 9. Inference Demo
We simulate the runtime API behavior. We load the model, read a raw features vector, compute the predicted values, and run the SHAP Explainer to generate local feature attribution values.

In [ ]:
# Load model and metadata
loaded_model_24h = lgb.Booster(model_file="backend/trained_models/lgbm_aqi_24h.txt")

with open("backend/trained_models/model_metadata.json", "r") as f:
    meta = json.load(f)
    meta_features = meta["features"]

# Select a single record for inference
sample_record = X_test.iloc[[0]]
actual_target = y_test_24h.iloc[0]

# Predict
pred_val = loaded_model_24h.predict(sample_record)[0]
print(f"Inference Sample Output:")
print(f"- Predicted 24h ahead AQI: {pred_val:.2f}")
print(f"- Actual 24h ahead AQI: {actual_target:.2f}")

# Explain
explainer = shap.TreeExplainer(loaded_model_24h)
shap_attribs = explainer(sample_record)

# Compile local contributions
contribs = []
for i, feat in enumerate(meta_features):
    val = sample_record[feat].values[0]
    sh_val = shap_attribs.values[0][i]
    contribs.append({"feature": feat, "value": val, "shap_value": sh_val})
    
df_contribs = pd.DataFrame(contribs).sort_values(by="shap_value", key=abs, ascending=False)
print("\nLocal Feature Contributions for this prediction (Top 5):")
print(df_contribs.head(5))